<a href="https://colab.research.google.com/github/praveenkumarreddyaturu104/Next-Word-Prediction-using-TensorFlow/blob/main/Next_Word_Prediction_using_TensorFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow nltk matplotlib

In [2]:
import tensorflow as tf
import numpy as np
import nltk
import matplotlib.pyplot as plt
import pickle
import heapq
import re

from nltk.tokenize import RegexpTokenizer

print("TensorFlow Version :", tf.__version__)

TensorFlow Version : 2.20.0


In [3]:
import requests

url = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"

text = requests.get(url).text.lower()

print("Corpus Length :", len(text))

Corpus Length : 593911


In [4]:
tokenizer = RegexpTokenizer(r'\w+')

words = tokenizer.tokenize(text)

print(words[:50])

['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'adventures', 'of', 'sherlock', 'holmes', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'united', 'states', 'and', 'most', 'other', 'parts', 'of', 'the', 'world', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', 'you', 'may', 'copy', 'it', 'give', 'it', 'away', 'or', 're', 'use', 'it']


In [5]:
unique_words = sorted(set(words))

print("Vocabulary Size :", len(unique_words))

word_to_index = {w:i for i,w in enumerate(unique_words)}
index_to_word = {i:w for i,w in enumerate(unique_words)}

Vocabulary Size : 8185


In [6]:
WORD_LENGTH = 5

prev_words = []
next_words = []

for i in range(len(words)-WORD_LENGTH):

    prev_words.append(words[i:i+WORD_LENGTH])
    next_words.append(words[i+WORD_LENGTH])

print(prev_words[0])
print(next_words[0])

['the', 'project', 'gutenberg', 'ebook', 'of']
the


In [7]:
X = np.zeros(
    (len(prev_words), WORD_LENGTH, len(unique_words)),
    dtype=np.bool_
)

Y = np.zeros(
    (len(next_words), len(unique_words)),
    dtype=np.bool_
)

for i,sentence in enumerate(prev_words):

    for t,word in enumerate(sentence):

        X[i,t,word_to_index[word]] = 1

    Y[i,word_to_index[next_words[i]]] = 1

print(X.shape)
print(Y.shape)

(109113, 5, 8185)
(109113, 8185)


In [8]:
model = tf.keras.Sequential([

    tf.keras.layers.LSTM(
        128,
        input_shape=(WORD_LENGTH,len(unique_words))
    ),

    tf.keras.layers.Dense(
        len(unique_words),
        activation="softmax"
    )

])

optimizer = tf.keras.optimizers.RMSprop(
    learning_rate=0.01
)

model.compile(

    optimizer=optimizer,

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │     4,256,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8185)           │     1,055,865 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,312,633 (20.27 MB)

 Trainable params: 5,312,633 (20.27 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X,Y,validation_split=0.05,batch_size=128,epochs=10,shuffle=True)

In [ ]:
model.save("next_word_model.keras")

with open("history.pkl","wb") as f:
    pickle.dump(history.history,f)

In [ ]:
model = tf.keras.models.load_model("next_word_model.keras")

with open("history.pkl","rb") as f:
    history = pickle.load(f)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history["accuracy"])

plt.plot(history["val_accuracy"])

plt.title("Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend(["Train","Validation"])

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history["loss"])

plt.plot(history["val_loss"])

plt.title("Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend(["Train","Validation"])

plt.show()

In [ ]:
def prepare_input(sentence):

    sentence = sentence.lower().split()

    x = np.zeros(
        (1,WORD_LENGTH,len(unique_words))
    )

    for t,word in enumerate(sentence):

        if word in word_to_index:

            x[0,t,word_to_index[word]] = 1

    return x

In [ ]:
def sample(preds,top_n=3):

    preds = np.asarray(preds).astype("float64")

    preds = np.log(preds+1e-8)

    exp_preds = np.exp(preds)

    preds = exp_preds/np.sum(exp_preds)

    return heapq.nlargest(
        top_n,
        range(len(preds)),
        preds.take
    )

In [ ]:
def predict_next_word(sentence):

    x = prepare_input(sentence)

    preds = model.predict(x,verbose=0)[0]

    next_index = sample(preds,1)[0]

    return index_to_word[next_index]

In [ ]:
def predict_top_words(sentence,n=5):

    x = prepare_input(sentence)
    preds = model.predict(x,verbose=0)[0]
    indices = sample(preds,n)
    return [index_to_word[i] for i in indices]

In [ ]:
quotes = [
"It is not a lack of love",
"That which does not kill",
"I'm not upset that you",
"And those who were seen",
"It is hard enough to"
]

for q in quotes:

    words = tokenizer.tokenize(q.lower())
    words = words[-5:]
    sentence = " ".join(words)
    print(sentence)
    print(predict_top_words(sentence,5))
    print()

In [ ]:
Next Word Prediction using TensorFlow

Description:

✔ Developed a Next Word Prediction Model by implementing an LSTM-based deep learning architecture using TensorFlow/Keras.
✔ Built an NLP preprocessing pipeline by tokenizing text, generating word vocabularies, and creating sequential datasets from the Project Gutenberg corpus.
✔ Trained and evaluated the model by applying RMSprop optimization and categorical cross-entropy loss.
✔ Generated context-aware next-word suggestions by performing inference on user-provided text sequences.
✔ Visualized model performance by plotting training and validation accuracy and loss metrics.